# Multivariate Regression with PyTorch

Solving a multivariate regression problem using a simple neural network in PyTorch. 

We will be covering the entire process from creating a synthetic dataset to `training a model`, `saving` it, and then `loading` it for `inference on new data`.

## 1. Imports

We'll use `torch` for building and training the neural network, `numpy` for numerical operations, and `matplotlib` for plotting.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader # for handling datasets
import numpy as np
import matplotlib.pyplot as plt

## 2. Data Generation

Let's assume we have 3 input features (`x1`, `x2`, `x3`) and one output feature (`y`). 

The relationship will be linear with some added noise to simulate a real-world scenario.

The underlying function is: `y = 2*x1 + 3*x2 - 5*x3 + 7 + noise`

In [ ]:
num_samples = 1000
num_features = 3

# Generate random input features
X = np.random.rand(num_samples, num_features) * 10

# Generate the output 'y' based on a linear combination of inputs plus some noise
noise = np.random.randn(num_samples, 1) * 1.5
y = 2 * X[:, 0:1] + 3 * X[:, 1:2] - 5 * X[:, 2:3] + 7 + noise

print("Shape of input features (X):", X.shape)
print("Shape of output (y):", y.shape)

In [ ]:
X, y[:10]

## 3. Data Preparation

Next, we convert our NumPy arrays into PyTorch tensors. We then create a `TensorDataset` and a `DataLoader` to handle batching and shuffling of the data during training.

1. **`TensorDataset`**: This is a PyTorch `Dataset` that wraps tensors. 

Each sample will be retrieved by indexing tensors along the first dimension. 

It's useful for simple datasets where all samples (features and labels) are already in tensors. 

[Official Documentation](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)

2. **`DataLoader`**: This utility helps in loading data in batches, shuffling it, and performing multiprocessing for data loading.

It abstracts away the complexity of data iteration, making it easier to feed data into your model during training. 

[Official Documentation](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

In [ ]:
# Convert NumPy arrays to PyTorch tensors
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).float()

# Create a dataset and dataloader
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

## Inspect dataset

In [ ]:
print("Type:", type(dataset))
print("Number of samples:", len(dataset))

if hasattr(dataset, "tensors"):
    for i, t in enumerate(dataset.tensors):
        print(f"Tensor {i}: shape={tuple(t.shape)}, dtype={t.dtype}")

print("\nFirst 3 samples (features, label):")
for i in range(3):
    x_i, y_i = dataset[i]
    print(f"Sample {i}: x.shape={tuple(x_i.shape)}, y.shape={tuple(y_i.shape)}; x={x_i.numpy()}, y={y_i.numpy()}")

## train_loader (uses dataset and train_loader defined earlier)


In [ ]:
print("Type:", type(train_loader))
try:
    n_batches = len(train_loader) # Number of batches per epoch
except TypeError:
    n_batches = None

print("Batches per epoch:", n_batches)
print(f"Train Loader : {train_loader}")

print("\nConfigured batch_size attribute:", getattr(train_loader, "batch_size", None))
print("num_workers:", getattr(train_loader, "num_workers", None))
print("Loader.dataset is dataset variable:", train_loader.dataset is dataset)

## Inspect one batch to show shapes and dtypes

In [ ]:
batch_inputs, batch_labels = next(iter(train_loader))
print("\nOne batch sample:")
print(" - inputs shape:", tuple(batch_inputs.shape), "dtype:", batch_inputs.dtype)
print(" - labels shape:", tuple(batch_labels.shape), "dtype:", batch_labels.dtype)

# Concise summary
n_samples = len(dataset)
n_features = batch_inputs.shape[1]
target_dim = batch_labels.shape[1] if batch_labels.dim() > 1 else 1
print(f"\nSummary: {n_samples} samples, each input has {n_features} features, target dim = {target_dim}.")
if n_batches is not None:
    print(f"DataLoader yields {n_batches} batches per epoch with batch_size = {train_loader.batch_size}.")

## 4. Model Definition

We'll define a simple neural network. For this linear problem, a single linear layer is sufficient, but we'll add a hidden layer to demonstrate a more general approach.

- **Input Layer**: Takes `num_features` (3) inputs.
- **Hidden Layer**: A hidden layer with 10 neurons and a ReLU activation function.
- **Output Layer**: Produces a single continuous value.

In [ ]:
class MultivariateRegressionModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(MultivariateRegressionModel, self).__init__()
        self.linear1 = nn.Linear(input_size, 10)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(10, output_size)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

# Instantiate the model
input_dim = num_features
output_dim = 1
model = MultivariateRegressionModel(input_dim, output_dim)
print(model)

In [ ]:
model.state_dict()

## 5. Training the Model

Now, we'll set up the training loop.

1.  **Loss Function**: We use Mean Squared Error (MSE) loss, which is common for regression tasks.
2.  **Optimizer**: We use the Adam optimizer to update the model's weights.
3.  **Training Loop**: We iterate through the data for a specified number of epochs, performing the forward pass, calculating the loss, and backpropagating to update the weights.

In [ ]:
learning_rate = 0.01
num_epochs = 100

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(train_loader):
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

## 6. Inference and Evaluation

After training, let's see how well our model performs. 

We'll use the entire dataset for prediction and compare the predicted values against the actual values.

In [ ]:
# Switch model to evaluation mode
model.eval()

# Make predictions
with torch.no_grad():
    predicted = model(X_tensor).detach().numpy()

In [ ]:
# Plot the results
plt.figure(figsize=(10, 6))
plt.scatter(y.ravel(), predicted.ravel(), alpha=0.5)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values")
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2) # Diagonal line
plt.grid(True)
plt.show()

## 7. Saving the Model

It's good practice to save your trained model. 

We save the model's `state_dict`, which contains all the learned weights and biases.

In [ ]:
MODEL_NAME = 'multivariate_regression_model.pth'
import os
if os.path.exists(MODEL_NAME):
    os.remove(MODEL_NAME)
    print("Existing Model removed successfully.")

torch.save(model.state_dict(), MODEL_NAME)
print("Model saved successfully.")

## 8. Loading the Model

Now, let's simulate a new session where we load our pre-trained model for inference. 

We first need to instantiate a new model with the same architecture and then load the saved `state_dict`.

In [ ]:
# Create a new instance of the model
loaded_model = MultivariateRegressionModel(input_dim, output_dim)

# Load the saved state dictionary
loaded_model.load_state_dict(torch.load(MODEL_NAME))

# Set the model to evaluation mode
loaded_model.eval() 

print("Model loaded successfully!")

In [ ]:
# Verify loaded model
loaded_model, loaded_model.state_dict()

## 9. Prediction on New Data

Finally, let's use our loaded model to make predictions on new, unseen data.

In [ ]:
# Create some new data points
new_data = np.array([
    [1.0, 2.0, 3.0],  # Expected y = 2*1 + 3*2 - 5*3 + 7 = 0
    [4.0, 5.0, 1.0],  # Expected y = 2*4 + 3*5 - 5*1 + 7 = 25
    [8.0, 1.0, 2.0],  # Expected y = 2*8 + 3*1 - 5*2 + 7 = 16
    [9.2, 2.6, 3.0]   
], dtype=np.float32)

# Convert to a PyTorch tensor
new_data_tensor = torch.from_numpy(new_data)

# Make predictions with the loaded model
with torch.no_grad():
    new_predictions = loaded_model(new_data_tensor)

print("New Data:")
print(new_data)
print("\nPredictions from loaded model:")
print(new_predictions.numpy())

## Plot the results for the new samples

In [ ]:
# Compute the true (actual) y for the new_data using the same underlying formula
expected_y = 2 * new_data[:, 0] + 3 * new_data[:, 1] - 5 * new_data[:, 2] + 7

# Convert predictions to a 1D numpy array
preds = new_predictions.numpy().squeeze()

plt.figure(figsize=(10, 6))
plt.scatter(expected_y, preds, alpha=0.6)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values (New Data)")
plt.plot([expected_y.min(), expected_y.max()], [expected_y.min(), expected_y.max()], 'k--', lw=2) # Diagonal line
plt.grid(True)
plt.show()

## Conclusion

- We have, generated a synthetic dataset with a known linear relation: y = 2*x1 + 3*x2 - 5*x3 + 7 + noise.
- Converted the data to PyTorch tensors and used `TensorDataset/DataLoader` for batching.
- Defined and trained a small neural network (`input -> hidden ReLU -> output`) using MSE loss and Adam optimizer.
- Training reduced the loss and the model learned to approximate the underlying mapping.
- `Saved the model state_dict` and successfully `reloaded it for inference`.
- Made `predictions` on new unseen samples `using saved model`.